In [3]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------

def sanity_check(adata, color=None):
    print("\n======================")
    print("🔍   ANNDATA CHECK")
    print("======================")
    print(f"Shape: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    print("obs columns:", list(adata.obs.columns))
    print("obsm keys:", list(adata.obsm.keys()))

    if color is not None and color in adata.obs:
        print(f"\nColumn '{color}': {adata.obs[color].nunique()} unique values")
        print(f"Missing: {adata.obs[color].isna().sum():,}")
    elif color is not None:
        print(f"\n⚠️ Column '{color}' NOT FOUND")

    if "X_umap" in adata.obsm:
        print("UMAP present:", adata.obsm["X_umap"].shape)
    else:
        print("No UMAP yet")
    print("======================\n")



def plot_umap(adata, color, palette=None, title=None, out_png=None):
    fig = sc.pl.umap(
        adata,
        color=color,
        title=title,
        frameon=False,
        palette=palette,
        show=False,
        return_fig=True,
    )
    plt.show()

    if out_png:
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"Saved: {out_png}")


cell_type_palette = {'ABCs': '#023fa5',
 'Astrocytes': '#7d87b9',
 'Astroependymal': '#bec1d4',
 'BAMs': '#d6bcc0',
 'Bergmann': '#bb7784',
 'Choroid-Plexus': '#8e063b',
 'ECs': '#4a6fe3',
 'Ependymal': '#8595e1',
 'Immune-Other': '#b5bbe3',
 'Microglia': '#e6afb9',
 'Neurons-Dopa': '#e07b91',
 'Neurons-Gaba': '#d33f6a',
 'Neurons-Glut': '#11c638',
 'Neurons-Glyc-Gaba': '#8dd593',
 'Neurons-Granule-Immature': '#c6dec7',
 'Neurons-Other': '#ead3c6',
 'OECs': '#f0b98d',
 'OPCs': '#ef9708',
 'Oligodendrocytes': '#0fcfc0',
 'Pericytes': '#9cded6',
 'SMCs': '#d5eae7',
 'Tanycytes': '#f3e1eb',
 'Undefined': '#f6c4e1',
 'VLMCs': '#f79cd4'}


In [1]:
# ---------------------------------------------------
# Paths
# ---------------------------------------------------
ATLAS_PATH = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/abc_atlas.h5ad"

# ---------------------------------------------------
# Load data
# ---------------------------------------------------
print(f"📂 Loading {ATLAS_PATH}")
adata = sc.read(ATLAS_PATH)

print("Data loaded:")
print(adata)
print("obs columns:", list(adata.obs.columns))




📂 Loading /p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/abc_atlas.h5ad
Data loaded:
AnnData object with n_obs × n_vars = 2349544 × 32285
    obs: 'abc_sample_id', 'anatomical_division_label', 'barcoded_cell_sample_label', 'brain_section_label', 'cell_barcode', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'dataset_label', 'donor_genotype', 'donor_label', 'donor_sex', 'entity', 'feature_matrix_label', 'library_label', 'library_method', 'neurotransmitter', 'neurotransmitter_color', 'region_of_interest_acronym', 'region_of_interest_color', 'region_of_interest_order', 'subclass', 'subclass_color', 'supertype', 'supertype_color', 'x', 'y'
obs columns: ['abc_sample_id', 'anatomical_division_label', 'barcoded_cell_sample_label', 'brain_section_label', 'cell_barcode', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'dataset_label', 'donor_genotype', 'donor_label', 'donor_sex', 'entity', 'feature_matrix_label', 'library_label', 'library_

In [4]:
atlas = adata
print("Shape:", atlas.shape)
print("obs columns:", list(atlas.obs.columns))

# ---------------------------------------------------
# Add cell_id if missing
# ---------------------------------------------------
if "cell_id" not in atlas.obs:
    atlas.obs["cell_id"] = atlas.obs.index.astype(str)

# ---------------------------------------------------
# Subsample to 100k
# ---------------------------------------------------
print("📉 Subsampling atlas to 100,000 cells…")
sc.pp.subsample(atlas, n_obs=100000, random_state=0)

print("New shape:", atlas.shape)

# ---------------------------------------------------
# PCA on transcriptomic X
# ---------------------------------------------------
print("📉 Running PCA on raw X…")
sc.tl.pca(atlas, n_comps=50, svd_solver="arpack")

# ---------------------------------------------------
# Neighbors + UMAP
# ---------------------------------------------------
print("👥 Computing neighbors…")
sc.pp.neighbors(atlas, use_rep="X_pca", n_neighbors=30)

print("🌀 Computing UMAP…")
sc.tl.umap(atlas, min_dist=0.3, random_state=0)


Shape: (2349544, 32285)
obs columns: ['abc_sample_id', 'anatomical_division_label', 'barcoded_cell_sample_label', 'brain_section_label', 'cell_barcode', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'dataset_label', 'donor_genotype', 'donor_label', 'donor_sex', 'entity', 'feature_matrix_label', 'library_label', 'library_method', 'neurotransmitter', 'neurotransmitter_color', 'region_of_interest_acronym', 'region_of_interest_color', 'region_of_interest_order', 'subclass', 'subclass_color', 'supertype', 'supertype_color', 'x', 'y']
📉 Subsampling atlas to 100,000 cells…
New shape: (100000, 32285)
📉 Running PCA on raw X…
👥 Computing neighbors…
🌀 Computing UMAP…


In [5]:
# ---------------------------------------------------
# Plot using your function
# ---------------------------------------------------
out_png = "atlas_umap_raw_100k.png"

plot_umap(
    atlas,
    color="sub_class",             
    palette=cell_type_palette,
    title="Atlas Transcriptomic UMAP (raw → PCA → UMAP, 100k)",
    out_png=out_png,
)

print("🎉 DONE — saved at:", out_png)

KeyError: 'Could not find key sub_class in .var_names or .obs.columns.'

In [ ]:
plot_umap(
    atlas,
    color="class",            
    #palette=cell_type_palette,
    title="Atlas Transcriptomic UMAP no palette",
    out_png=out_png,
)

print("🎉 DONE — saved at:", out_png)

In [ ]:
atlas.obs["class"].unique().tolist()
